# **Baseline Models para Previsão de Churn** 🤖

# 📏 1. Definição das Métricas de Avaliação

Nesta etapa, são definidas as métricas técnicas e de negócio que serão utilizadas para avaliar o desempenho dos modelos baseline no problema de churn.

💡 **Justificativa das métricas:**

Como a variável alvo apresenta desbalanceamento entre clientes que permaneceram e clientes que realizaram churn, a análise do desempenho dos modelos não será baseada apenas em accuracy.

Serão utilizadas métricas como **precision, recall, F1-score, ROC-AUC e PR-AUC**, pois elas permitem avaliar melhor a capacidade do modelo de identificar corretamente os clientes com maior risco de churn.

O **recall** será especialmente importante neste contexto, já que representa a proporção de clientes que realmente cancelaram e que foram corretamente identificados pelo modelo. Já o **F1-score** será utilizado como uma métrica de equilíbrio entre precision e recall.

Além disso, métricas como **ROC-AUC** e **PR-AUC** serão empregadas para comparar a capacidade de separação entre as classes, sendo a PR-AUC particularmente útil em cenários com desbalanceamento.

In [5]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [6]:
def avaliar_modelo(nome_modelo, y_true, y_pred, y_prob):
    resultados = {
        "modelo": nome_modelo,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob)
    }

    print(f"Modelo: {nome_modelo}")
    print(f"Accuracy:  {resultados['accuracy']:.4f}")
    print(f"Precision: {resultados['precision']:.4f}")
    print(f"Recall:    {resultados['recall']:.4f}")
    print(f"F1-score:  {resultados['f1_score']:.4f}")
    print(f"ROC-AUC:   {resultados['roc_auc']:.4f}")
    print(f"PR-AUC:    {resultados['pr_auc']:.4f}")

    return resultados

In [7]:
# Teste simples da função
y_true = [0, 1, 0, 1, 1]
y_pred = [0, 1, 0, 0, 1]
y_prob = [0.1, 0.8, 0.2, 0.3, 0.9]

resultado_teste = avaliar_modelo("Teste", y_true, y_pred, y_prob)

Modelo: Teste
Accuracy:  0.8000
Precision: 1.0000
Recall:    0.6667
F1-score:  0.8000
ROC-AUC:   1.0000
PR-AUC:    1.0000


💡 **Conclusão da etapa de métricas:**

Considerando o objetivo de identificar clientes com maior risco de churn, serão priorizadas métricas que avaliem corretamente a capacidade de detecção da classe positiva, especialmente **recall**, **F1-score**, **ROC-AUC** e **PR-AUC**. Essas métricas serão utilizadas nas próximas etapas para comparar os modelos baseline e verificar se eles apresentam desempenho superior a um classificador ingênuo.

# 🧹 2. Preparação dos Dados

Nesta etapa, os dados serão preparados para a construção dos modelos baseline, incluindo seleção da variável alvo, separação entre variáveis explicativas e target, divisão entre treino e teste e definição do pré-processamento.

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [10]:
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [11]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

In [12]:
df['Total Charges'].isnull().sum()

np.int64(11)

In [14]:
y = df['Churn Value']
y.value_counts()

Churn Value
0    5174
1    1869
Name: count, dtype: int64

In [15]:
colunas_excluir = [
    'CustomerID',
    'Count',
    'Country',
    'State',
    'Lat Long',
    'Latitude',
    'Longitude',
    'Churn Label',
    'Churn Value',
    'Churn Score',
    'CLTV',
    'Churn Reason'
]

X = df.drop(columns=colunas_excluir)

In [16]:
print("Dimensão de X:", X.shape)
print("Dimensão de y:", y.shape)

X.head()

Dimensão de X: (7043, 21)
Dimensão de y: (7043,)


,City,Zip Code,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,...,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges
0,Los Angeles,90003,Male,No,No,No,2,Yes,No,DSL,...,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
1,Los Angeles,90005,Female,No,No,Yes,2,Yes,No,Fiber optic,...,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65
2,Los Angeles,90006,Female,No,No,Yes,8,Yes,Yes,Fiber optic,...,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50
3,Los Angeles,90010,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,...,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05
4,Los Angeles,90015,Male,No,No,Yes,49,Yes,Yes,Fiber optic,...,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30


In [17]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'string']).columns.tolist()

print("Colunas numéricas:")
print(numeric_features)

print("\nColunas categóricas:")
print(categorical_features)

Colunas numéricas:
['Zip Code', 'Tenure Months', 'Monthly Charges', 'Total Charges']

Colunas categóricas:
['City', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [19]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5634, 21)
X_test: (1409, 21)
y_train: (5634,)
y_test: (1409,)


In [20]:
from sklearn.impute import SimpleImputer

In [21]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [22]:
print("Pré-processamento configurado com sucesso.")

Pré-processamento configurado com sucesso.


print("Pré-processamento configurado com sucesso.")